In [1]:
pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.4 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Use the function to set the seed
set_seed(42)


In [41]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv, global_mean_pool

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

dataset = KarateClub()
data = dataset[0]

num_train_nodes = int(0.8 * data.num_nodes)
data.train_mask = torch.zeros(data.num_nodes, dtype=bool)
data.train_mask[:num_train_nodes] = True
data.test_mask = ~data.train_mask

class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.out_channels = out_channels

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.bn2(self.conv2(x, edge_index))
        return x

class GNN_KNN_MLP(nn.Module):
    def __init__(self, gnn, mlp_hidden_dim, num_classes, k=3):
        super(GNN_KNN_MLP, self).__init__()
        self.gnn = gnn
        self.k = k
        self.pool = global_mean_pool
        self.mlp = nn.Sequential(
            nn.Linear(gnn.out_channels, mlp_hidden_dim),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(mlp_hidden_dim, 1),
        )
        self.classifier = nn.Linear(gnn.out_channels, num_classes)

    def forward(self, x, edge_index, batch):
        embeddings = self.gnn(x, edge_index)
        distances = torch.cdist(embeddings, embeddings)
        knn_indices = torch.topk(-distances, self.k + 1, dim=-1)[1][:, 1:]

        out_logits = torch.zeros((embeddings.size(0), self.classifier.out_features), device=x.device)
        for i in range(embeddings.size(0)):
            knn_set = torch.cat((embeddings[knn_indices[i]], embeddings[i].unsqueeze(0)), dim=0)
            pooled_embedding = self.pool(knn_set, batch=None)

            include_prob = torch.sigmoid(self.mlp(pooled_embedding)).view(-1)

            inclusion_sample = (torch.rand_like(include_prob) < include_prob).float()
            #print('inclusion_sample ',inclusion_sample)

            straight_through_sample = inclusion_sample + (include_prob - include_prob.detach())

            #print('straigth_sample ',straight_through_sample)

            selected_embedding = straight_through_sample * pooled_embedding + (1 - straight_through_sample) * embeddings[i]

            if inclusion_sample.item() == 1.0:
                print(f"Node {i}: Using pooled embedding.")
            else:
                print(f"Node {i}: Using original node embedding.")

            out_logits[i] = self.classifier(selected_embedding)

        return F.log_softmax(out_logits, dim=-1)

num_classes = data.y.max().item() + 1
gnn = GNN(in_channels=dataset.num_node_features, hidden_channels=16, out_channels=8)
model = GNN_KNN_MLP(gnn, mlp_hidden_dim=16, num_classes=num_classes, k=3)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.batch)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
    optimizer.step()

    # Logging gradients
    # for name, param in model.named_parameters():
    #     if param.grad is not None:
    #         print(f"{name} gradient: {param.grad}")

    #print(f"Training Loss: {loss.item()}")
    return loss.item()

def test():
    model.eval()
    correct = 0
    out = model(data.x, data.edge_index, data.batch)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum().item()
    test_size = data.test_mask.sum().item()
    accuracy = correct / test_size if test_size > 0 else 0.0
    return accuracy

for epoch in range(1, 1000):
    train_loss = train()
    test_acc = test()
    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Test Accuracy = {test_acc:.4f}\n")


Epoch 1: Train Loss = 1.5407, Test Accuracy = 0.0000

Epoch 2: Train Loss = 1.3762, Test Accuracy = 0.0000

Epoch 3: Train Loss = 1.2236, Test Accuracy = 0.0000

Epoch 4: Train Loss = 1.1405, Test Accuracy = 0.0000

Epoch 5: Train Loss = 1.0026, Test Accuracy = 0.0000

Epoch 6: Train Loss = 0.9137, Test Accuracy = 0.0000

Epoch 7: Train Loss = 0.8425, Test Accuracy = 0.0000

Epoch 8: Train Loss = 0.7511, Test Accuracy = 0.0000

Epoch 9: Train Loss = 0.7128, Test Accuracy = 0.0000

Epoch 10: Train Loss = 0.6649, Test Accuracy = 0.0000

Epoch 11: Train Loss = 0.6240, Test Accuracy = 0.0000

Epoch 12: Train Loss = 0.5968, Test Accuracy = 0.0000

Epoch 13: Train Loss = 0.5482, Test Accuracy = 0.0000

Epoch 14: Train Loss = 0.5315, Test Accuracy = 0.0000

Epoch 15: Train Loss = 0.5164, Test Accuracy = 0.0000

Epoch 16: Train Loss = 0.4640, Test Accuracy = 0.0000

Epoch 17: Train Loss = 0.4451, Test Accuracy = 0.0000

Epoch 18: Train Loss = 0.4064, Test Accuracy = 0.0000

Epoch 19: Train Los

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Epoch 206: Train Loss = 0.0043, Test Accuracy = 0.7143

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-41-1515f669633a>", line 128, in <cell line: 126>
    test_acc = test()
  File "<ipython-input-41-1515f669633a>", line 119, in test
    out = model(data.x, data.edge_index, data.batch)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
  File "<ipython-input-41-1515f669633a>", line 68, in forward
    include_prob = torch.sigmoid(self.mlp(pooled_embedding)).view(-1)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
   

TypeError: object of type 'NoneType' has no len()

tensor([1, 1, 1, 1, 3, 3, 3, 1, 0, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0,
        2, 2, 0, 0, 2, 0, 0, 2, 0, 0])